# Exercise: Storing Wikipedia Data in Redis

In this exercise you will download the [list of programming languages](https://en.wikipedia.org/wiki/List_of_programming_languages) from Wikipedia and store it in Redis, using different Redis data types.

**What you will practice:** hashes, sets, counters and TTL. Look at the [Redis introduction notebook](redis_intro.ipynb) if you need a reminder of the commands.

Complete the cells marked with `# YOUR CODE HERE`.

## Setup
The first cells install the libraries, start Redis on Colab (or use fakeredis elsewhere) and connect to it.

In [ ]:
%pip install -q redis fakeredis requests beautifulsoup4

In [ ]:
import subprocess, sys, time

# On Google Colab we can install and start a real Redis server.
# Elsewhere we use a Redis server running on your computer, if there is one (see the next cell).
if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq install -y redis-server > /dev/null", shell=True, check=True)
    subprocess.run("redis-server --daemonize yes", shell=True, check=True)
    time.sleep(1)

In [ ]:
import redis
import fakeredis

_fake_server = fakeredis.FakeServer()

def connect(db=0):
    """Connect to the Redis server on this machine, or to fakeredis if no server is running."""
    r = redis.Redis(host="localhost", port=6379, db=db, decode_responses=True)
    try:
        r.ping()
        return r
    except redis.ConnectionError:
        return fakeredis.FakeRedis(server=_fake_server, db=db, decode_responses=True)

r = connect()
r.flushdb()  # start from an empty database
print("Connected to:", type(r).__name__)

## Getting the data
The function below downloads the Wikipedia page and returns a list of dictionaries with the name of each language and the link to its page.

Wikipedia asks scripts to identify themselves with a `User-Agent` header, so we send one. If the structure of the page changes in the future, the selector `.div-col li` may need to be updated.

In [ ]:
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

def scrape_programming_languages():
    url = "https://en.wikipedia.org/wiki/List_of_programming_languages"
    headers = {"User-Agent": "BigDataCourse/1.0 (educational exercise)"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    languages = []
    for li in soup.select(".div-col li"):
        a = li.find("a")
        languages.append({
            "language": li.text.strip(),
            "link": urljoin(url, a["href"]) if a and a.get("href") else "No link",
        })
    return languages

languages = scrape_programming_languages()
print(len(languages), "languages found")
for lang in languages[:5]:
    print(lang)

## Task 1: Store each language as a hash
Store every language in a hash with key `language:<n>` (where `<n>` goes from 1 to the number of languages) and two fields: `name` and `link`.

Then print the hash `language:1` to check your work.

*Hint:* `r.hset(key, mapping={...})`

In [ ]:
# YOUR CODE HERE

## Task 2: Keep a counter
Store the total number of languages in the key `languages:count`. Then simulate adding a new language by incrementing the counter **atomically**, and print the new value.

*Hint:* `INCR`

In [ ]:
# YOUR CODE HERE

## Task 3: Index by first letter
For each language, add its id (`<n>`) to a **set** named `letter:<first letter>` (for example `letter:P` for Python).

Then:
1. print how many languages start with `P`
2. print the names of all the languages that start with `Z`

*Hint:* `SADD`, `SCARD`, `SMEMBERS`. To get the name from an id, use `HGET`.

In [ ]:
# YOUR CODE HERE

## Task 4: Which letter is the most popular?
Using the sets of Task 3, find the 5 letters with the most languages.

*Hint:* you can loop over the letters, or store the counts in a **sorted set** (`ZADD`) and read the top 5 with `ZREVRANGE`.

In [ ]:
# YOUR CODE HERE

## Task 5: Cache the download
Downloading the page every time is slow and puts load on Wikipedia. Write a function `get_languages()` that:
1. checks if the key `cache:languages` exists
2. if yes, returns the list stored there (as JSON)
3. if not, calls `scrape_programming_languages()`, stores the result in `cache:languages` as JSON **with a TTL of 1 hour**, and returns it

Call it twice and measure the time of each call. Which one is faster, and why?

*Hint:* `json.dumps`, `json.loads`, `r.set(key, value, ex=...)`, `time.perf_counter()`

In [ ]:
# YOUR CODE HERE